In [ ]:
"""
%pip install qiskit-ibm-runtime==0.47.0
%pip install samplomatic==0.18.0
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install pyswarms==1.3.0
%pip install pylatexenc==2.10
%pip install qiskit==2.3.0
%pip install numpy==2.4.2
%pip install pyscf==2.12.1
"""

'\n%pip install qiskit-ibm-runtime==0.47.0\n%pip install samplomatic==0.18.0\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install pyswarms==1.3.0\n%pip install pylatexenc==2.10\n%pip install qiskit==2.3.0\n%pip install numpy==2.4.2\n%pip install pyscf==2.12.1\n'

In [ ]:
import os
import time
import json
import logging
import numpy as np
import pandas as pd
from scipy.stats import qmc
import pyswarms.backend as P
from qiskit_aer import AerSimulator
from joblib import Parallel, delayed
from scipy.optimize import OptimizeResult
from qiskit_ibm_runtime import EstimatorV2
from pyswarms.backend.topology import Star
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_ibm_runtime.fake_provider import FakeBoston
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

2026-08-09 02:50:38,437 - qiskit.passmanager.base_tasks - INFO - Pass: UnrollCustomDefinitions - 0.09513 (ms)
2026-08-09 02:50:38,438 - qiskit.passmanager.base_tasks - INFO - Pass: BasisTranslator - 0.01740 (ms)
/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


In [ ]:
logging.getLogger('qiskit_nature').setLevel(logging.WARNING)

# 1°: Building the Molecular Problem


In [ ]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees,

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0, 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [ ]:
# The original space has approximately 6 electrons, 6 space orbitals, and 12
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)

reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

# 2°: The Hamiltonian in Terms of Qubits

In [ ]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [ ]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()

qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

In [ ]:
# Finally, we obtain the number of qubits and operator in terms of Pauli matrices.
num_qubits = qubit_op.num_qubits
print(f"Number of qubits = {num_qubits}")
print(f"Hamiltonian: {qubit_op}")

Number of qubits = 6
Hamiltonian: SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIIZZ', 'IIIZII', 'IIIZIZ', 'IIZIII', 'IIZIIZ', 'IZIIII', 'IZIIIZ', 'ZIIIII', 'ZIIIIZ', 'IYYIYY', 'IXXIYY', 'IYYIXX', 'IXXIXX', 'YZYYZY', 'XZXYZY', 'YZYXZX', 'XZXXZX', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'YYIYYI', 'XXIYYI', 'YYIXXI', 'XXIXXI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[-2.86740153+0.j,  0.32161792+0.j,  0.31034865+0.j,  0.06196671+0.j,
  0.12889394+0.j,  0.08001574+0.j,  0.32161792+0.j,  0.09975793+0.j,
  0.31034865+0.j,  0.10309655+0.j,  0.12889394+0.j,  0.09238729+0.j,
  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,
  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,
  0.08557179+0.j,  0.10309655+0.j,  0.10888574+0.j,  0.08924797+0.j,
  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,
  0.09238729+0.j,  0.08924797+0.j,  0.11246476+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.08557179+0.j])


# 3°: Ansatz Circuit Construction


In [ ]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [ ]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

# 4°: Transpilation and Simulator Settings

In [ ]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend, initial_layout, seed=14):
  target = backend.target

  pm = generate_preset_pass_manager(
       target=target,
       initial_layout=initial_layout,
       optimization_level=3,
       seed_transpiler=seed,
       layout_method='sabre',
       routing_method='sabre'
  )

  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

# 5°: Measuring Eigenvalues and Energies

In [ ]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 6°: Creating a custom Global Best PSO loop

The custom GPSO class will allow us to directly implement the GPSO optimizer as if we were implementing a class from qiskit algorithms/machine learning or from scipy itself.

In [ ]:
class GPSO:
    def __init__(self, maxiter: int, n_particles: int, dimensions: int, options: dict, bounds=(-np.pi, np.pi), callback=None, seed: int = None):
        self.maxiters = maxiter
        self.callback = callback
        self.n_particles = n_particles
        self.dimensions = dimensions
        self.options = options
        self.bounds = bounds
        self.seed = seed

        # Initializes the local random number generator (RNG).
        self.rng = np.random.default_rng(seed)

        # Global topology configuration (Star topology)
        self.topology = Star()
        self.swarm = P.create_swarm(n_particles=n_particles, dimensions=dimensions, options=options)

    def minimize(self, fun, x0=None):
        nit = 0
        nfev = 0
        lower_b, upper_b = self.bounds

        # Synchronizes the global NumPy state to lock the stochasticity of the pyswarms backend.
        if self.seed is not None:
            np.random.seed(self.seed)

        # 1. HALTON SEQUENCE INITIALIZATION
        # Initialize the Halton sampler for the problem dimensions
        # We passed the self.rng file to ensure that the Halton sequence is reproducible.
        sampler = qmc.Halton(d=self.dimensions, scramble=True, seed=self.rng)
        sample = sampler.random(n=self.n_particles)

        # Scale the sample from [0, 1] to the specified bounds [lower_b, upper_b]
        self.swarm.position = qmc.scale(sample, lower_b, upper_b)

        # 2. INSERTION OF x0 (e.g., Hartree-Fock state)
        if x0 is not None:
            self.swarm.position[0] = np.array(x0)

        # Synchronize initial memory (pbest) with generated positions
        self.swarm.pbest_pos = self.swarm.position.copy()
        self.swarm.pbest_cost = np.full(self.n_particles, np.inf, dtype=float)
        self.swarm.best_cost = np.inf
        self.swarm.best_pos = self.swarm.position[0].copy()

        # Bounds for the w decay
        # Fetches the initial 'w' value passed in the 'options' dictionary (e.g., 0.8)
        w_max = self.options.get('w', 0.8)
        w_min = 0.4

        # 3. OPTIMIZATION LOOP
        for i in range(self.maxiters):
            # --- STOCHASTIC LINEAR DECAY OF W ---
            # Linear base: smoothly decays from w_max (from your dict) to w_min over the iterations
            w_linear = w_max - ((w_max - w_min) * (i / self.maxiters))
            # Stochastic component: adds a small random variation (-0.05 to 0.05)
            # Helps particles escape false local minima caused by simulator noise
            w_stochastic = w_linear + self.rng.uniform(-0.05, 0.05)
            # Dynamically updates the swarm dictionary while keeping your other keys (c1, c2) intact
            self.swarm.options['w'] = w_stochastic

            costs = []
            # Evaluate each particle's cost individually
            for particle_position in self.swarm.position:
                cost = fun(particle_position)
                costs.append(cost)
                nfev += 1

            self.swarm.current_cost = np.asarray(costs, dtype=float)

            # PySwarms topological updates
            self.swarm.pbest_pos, self.swarm.pbest_cost = P.compute_pbest(self.swarm)
            self.swarm.best_pos, self.swarm.best_cost = self.topology.compute_gbest(self.swarm)

            self.swarm.velocity = self.topology.compute_velocity(self.swarm)
            self.swarm.position = self.topology.compute_position(self.swarm)

            # Boundary Handling: Ensure particles stay within physical limits
            self.swarm.position = np.clip(self.swarm.position, lower_b, upper_b)

            nit += 1

            # Native callback execution
            if self.callback is not None:
                self.callback(self.swarm.best_pos, self.swarm.best_cost)

        # Return the result in a SciPy-compatible OptimizeResult format
        result = OptimizeResult(
            fun=float(self.swarm.best_cost),
            x=np.array(self.swarm.best_pos, dtype=float),
            nit=nit,
            nfev=nfev,
            success=True,
            message="GPSO with Halton Initialization terminated successfully."
        )

        return result

# 7°: Configuring Parallel Execution Function

In [ ]:
def parallel_optimization(
    run_idx: int,
    seed_sim: int,
    seed_ri: int,
    gpso_seed: int,
    shots: int,
    maxiter: int,
    initial_layout:list
):

  # Initialization of the Aer simulator with FakeProvider
  backend = AerSimulator.from_backend(FakeBoston())
  backend.options.seed_simulator = seed_sim

  # Ansatz transpilation and observables for the ISA
  ansatz_isa, isa_observables = transpile_to_isa(
                                backend,
                                initial_layout,
                                )

  # Declaring and configuring the Estimator for calculating expected values.
  estimator = EstimatorV2(mode=backend)
  estimator.options.default_shots = shots

  # The energies and parameters per iteration will be stored here.
  energy_data = []
  params_data = []

  # Initializing random parameters between -1 to 1.
  rng = np.random.default_rng(seed_ri)
  initial_params = rng.uniform(-1, 1, ansatz_isa.num_parameters)

  # Callback function declaration
  def optimizer_callback(params, value):
    energy_data.append(value)
    params_data.append(params.tolist())

  # Statement of the cost function
  def energy_cost_function(params):
    estimator_job = estimator.run([(ansatz_isa, isa_observables, params)])
    estimator_exp_val = estimator_job.result()[0].data.evs
    return float(estimator_exp_val)

  # Optimizer instance with settings
  optimizer = GPSO(
              maxiter=maxiter,
              n_particles=20,
              dimensions=ansatz_isa.num_parameters,
              bounds=(-np.pi, np.pi),
              options={'c1': 1.43, 'c2': 1.43, 'w': 0.9},
              callback=optimizer_callback,
              seed=gpso_seed
            )

  # The optimization process takes place here.
  t0 = time.perf_counter()
  result = optimizer.minimize(fun=energy_cost_function, x0=initial_params)
  t1 = time.perf_counter()

  return {
        "optimizer": "GPSO",
        "run": run_idx,
        "seed_sim": seed_sim,
        "seed_ri": seed_ri,
        "gpso_seed": gpso_seed,
        "steps_requested": maxiter,
        "steps_done": result.nit,
        "cost_function_evaluation": result.nfev,
        "energies_trajectory_len": len(energy_data),
        "execution_time": t1 - t0,
        "final_energy": float(result.fun),
        "optimal_params": result.x.tolist(),
        "energies_trajectory": energy_data,
        "params_trajectory": params_data,
  }

# 8°: Running VQE in parallel on CPU cores

In [ ]:
maxiter = 1000
shots = 10_000
runs = 25
seed_sim = 42
seed_ri = 127
gpso_seed = 212
initial_layout= [43, 42, 56, 63, 62, 61]

In [ ]:
n_jobs = 25
print(f"Running {runs} GPSO runs in parallel (CPU) | n_jobs={n_jobs} | shots={shots} | iteration={maxiter}")

Running 25 GPSO runs in parallel (CPU) | n_jobs=25 | shots=10000 | iteration=1000


In [17]:
results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
    delayed(parallel_optimization)(
        run_idx=i,
        seed_sim=seed_sim+i,
        seed_ri=seed_ri+i,
        gpso_seed=gpso_seed+i,
        shots=shots,
        maxiter=maxiter,
        initial_layout=initial_layout
    )
    for i in range(runs)
)

[Parallel(n_jobs=25)]: Using backend LokyBackend with 25 concurrent workers.
[Parallel(n_jobs=25)]: Done   3 out of  25 | elapsed: 1190.5min remaining: 8730.5min
[Parallel(n_jobs=25)]: Done   6 out of  25 | elapsed: 1192.3min remaining: 3775.5min
[Parallel(n_jobs=25)]: Done   9 out of  25 | elapsed: 1194.2min remaining: 2123.0min
[Parallel(n_jobs=25)]: Done  12 out of  25 | elapsed: 1196.0min remaining: 1295.6min
[Parallel(n_jobs=25)]: Done  15 out of  25 | elapsed: 1197.8min remaining: 798.6min
[Parallel(n_jobs=25)]: Done  18 out of  25 | elapsed: 1199.4min remaining: 466.4min
[Parallel(n_jobs=25)]: Done  21 out of  25 | elapsed: 1200.4min remaining: 228.6min
[Parallel(n_jobs=25)]: Done  25 out of  25 | elapsed: 1204.2min finished


In [18]:
results.sort(key=lambda d: d["run"])

# 9°: Adding the constant energies

In [19]:
for run_data in results:

    raw_final_energy = run_data["final_energy"]
    true_final_energy = float(interpret_exp_val(raw_final_energy, reduced_molecule_problem))

    run_data["final_energy"] = true_final_energy

    raw_trajectory = run_data["energies_trajectory"]

    true_trajectory = [
        interpret_exp_val(step_energy, reduced_molecule_problem)
        for step_energy in raw_trajectory
    ]

    run_data["energies_trajectory"] = true_trajectory

# 10º Save the data for later analysis.

In [22]:
df = pd.DataFrame(results)

df["optimal_params_json"] = df["optimal_params"].apply(json.dumps)
df["energies_trajectory_json"] = df["energies_trajectory"].apply(json.dumps)
df["params_trajectory_json"] = df["params_trajectory"].apply(json.dumps)

df.to_csv(
    "BEH2_VQE_GPSO_HARDWARE_NOISE_1_RANDOM_INIT.csv",
    columns=[
        "optimizer",
        "run",
        "seed_sim",
        "seed_ri",
        "gpso_seed",
        "steps_requested",
        "steps_done",
        "cost_function_evaluation",
        "energies_trajectory_len",
        "execution_time",
        "final_energy",
        "optimal_params_json",
        "energies_trajectory_json",
        "params_trajectory_json",
    ],
    index=False,
)

print("\nSaved: BEH2_VQE_GPSO_HARDWARE_NOISE_1_RANDOM_INIT.csv")
print("Done.")


Saved: BEH2_VQE_GPSO_HARDWARE_NOISE_1_RANDOM_INIT.csv
Done.
